# Simple ML Pipeline Template

This template provides a simple example of a machine learning pipeline using Kubeflow Pipelines (KFP). It includes functions to:

- Create a dataset
- Preprocess the dataset
- Train a simple machine learning model
- Run inference using the trained model

The pipeline is designed to be customizable, allowing you to modify each function to fit your specific use case.

## Steps to Use the Template

### 1. Dataset Creation (`create_dataset`)
**Objective:** Create and save a dataset to a shared location.

**Modify the Code:** Customize the `create_dataset` function to fit your dataset creation process (e.g., loading from a file or generating synthetic data).

**Parameters:**
- `data_path`: Path to store the created dataset. Change this if needed.

**Action:** Once the function is modified, the dataset will be saved as a `.npy` file.

---

### 2. Data Preprocessing (`preprocess_data`)
**Objective:** Process the dataset created in the previous step. This can include tasks like normalization, feature engineering, or cleaning.

**Modify the Code:** Adjust the `preprocess_data` function to implement your preprocessing steps.

**Parameters:**
- `data_path`: Path to the dataset created in step 1.
- `processed_data_path`: Path to store the processed dataset. Change this if needed.

**Action:** The function will save the processed data to the specified path.

---

### 3. Model Training (`train_model`)
**Objective:** Train a machine learning model using the preprocessed dataset.

**Modify the Code:** Replace the simple linear model with your own model architecture and training process.

**Parameters:**
- `processed_data_path`: Path to the preprocessed data.
- `model_path`: Path to save the trained model.
- `log_dir`: Directory for TensorBoard logs (optional).

**Action:** After training, the model is saved in the `model_path` directory and logs are saved for TensorBoard.

---

### 4. Inference (`run_inference`)
**Objective:** Use the trained model to make predictions.

**Modify the Code:** Adjust the `run_inference` function to fit the input and output format of your model. For instance, modify the input data and the model prediction logic.

**Parameters:**
- `model_path`: Path to load the trained model.

**Action:** The model will be loaded, and predictions will be printed to the console.

---

### 5. Kubeflow Pipeline Structure (`simple_ml_pipeline`)
**Objective:** Define the sequence of steps in the machine learning pipeline.

**Modify the Code:**
- Link the components in the desired order. For example, the `train_model` task should follow the `preprocess_data` task.
- Define shared data volumes using Persistent Volume Claims (PVCs) to share data across components.

**Action:** The pipeline will execute the tasks in order, and each task will have access to the shared volume for reading/writing data.

---

### 6. Compile and Deploy the Pipeline  
**Objective:** Once the pipeline is defined, you need to compile it into a YAML or TAR file to upload to Kubeflow.  

**Action:**  
- Use `kfp.compiler.Compiler().compile()` to compile the pipeline into a `.yaml` file.  
- Upload the compiled `.yaml` file to your Kubeflow instance and run it as a pipeline.  
- **After uploading the pipeline**, create a **run** to execute the pipeline.  
- Ensure that a **Persistent Volume Claim (PVC)** is created with the same name as specified in the pipeline (e.g., `"shared-pvc"`) to provide shared storage for the pipeline components.  
- This PVC that you should craet manually will be assigned to the pipeline so that data can be shared across different steps.  

---

## Example Usage
1. Modify the functions (`create_dataset`, `preprocess_data`, `train_model`, `run_inference`) to suit your specific dataset, preprocessing steps, model architecture, and inference logic.
2. Customize the paths used for data storage (`data_path`, `processed_data_path`, `model_path`, etc.).
3. Define the pipeline structure by linking the components in the correct order using the `@dsl.pipeline` decorator.
4. Compile the pipeline and upload it to your Kubeflow instance to run.

---

## Requirements
Ensure the following Python packages are installed:

```bash
pip install kfp tensorflow numpy mlflow kubernetes boto3
```
**Remember that this cod eis running with the KFp v1.8.22**

This template should provide a good starting point for building machine learning pipelines in Kubeflow. Modify the dataset, model, and pipeline structure as needed to adapt to your specific use case.


In [4]:
import kfp
from kfp import dsl
from kfp.components import create_component_from_func
from kubernetes import client as k8s
import os
import tensorflow as tf
import mlflow
import numpy as np

# Print KFP version
print(kfp.__version__)


# Create a simple dataset function
def create_dataset():
    """
    Put your code for creating a dataset here.
    This is a helper function where you can define how your dataset will be created,
    including any preprocessing or transformations before saving it.
    Set the path where you want to save the dataset (data_path).
    """
    import os
    import numpy as np

    data_path = "/mnt/shared/data"  # Modify this path if needed
    np.random.seed(42)
    x = np.linspace(0, 2 * np.pi, 100)
    y = np.sin(x)
    dataset = np.column_stack((x, y))
    os.makedirs(data_path, exist_ok=True)
    np.save(os.path.join(data_path, "dataset.npy"), dataset)
    print(f"Dataset saved at {data_path}")


# Preprocess the dataset function
def preprocess_data():
    """
    Put your data preprocessing code here.
    This function processes the dataset you created in the previous step.
    Set the correct paths for your dataset and processed data.
    """
    import os
    import numpy as np

    data_path = "/mnt/shared/data"  # Modify this path if needed
    processed_data_path = "/mnt/shared/processed_data"  # Modify this path if needed
    dataset = np.load(os.path.join(data_path, "dataset.npy"))
    x, y = dataset[:, 0], dataset[:, 1]
    x_norm = (x - np.min(x)) / (np.max(x) - np.min(x))  # Normalize the data
    os.makedirs(processed_data_path, exist_ok=True)
    np.save(os.path.join(processed_data_path, "processed_dataset.npy"), np.column_stack((x_norm, y)))
    print("Preprocessed dataset saved.")


# Train a simple linear model function
def train_model():
    """
    Put your model training code here.
    This function trains a model using the preprocessed dataset.
    Set paths for saving the model and logs.
    """
    import os
    import tensorflow as tf
    import mlflow
    import numpy as np

    processed_data_path = "/mnt/shared/processed_data"  # Modify this path if needed
    model_path = "/mnt/shared/model"  # Modify this path if needed
    log_dir = "/mnt/shared/logs"  # Modify this path if needed

    mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
    mlflow.start_run()
    data = np.load(os.path.join(processed_data_path, "processed_dataset.npy"))
    x, y = data[:, 0], data[:, 1]

    model = tf.keras.Sequential([tf.keras.layers.Dense(1, input_shape=[1])])
    model.compile(optimizer="adam", loss="mse")

    tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir)
    model.fit(x, y, epochs=5, verbose=2, callbacks=[tensorboard_callback])

    os.makedirs(model_path, exist_ok=True)
    model.save(os.path.join(model_path, "simple_model.h5"))

    mlflow.log_param("epochs", 5)
    mlflow.log_artifact(os.path.join(model_path, "simple_model.h5"))
    mlflow.end_run()


# Inference from the trained model function
def run_inference():
    """
    Put your code for running inference here.
    This function uses the trained model to make predictions based on input data.
    Set the correct model path and data for prediction.
    """
    import os
    import tensorflow as tf
    import numpy as np

    model_path = "/mnt/shared/model"  # Modify this path if needed
    model = tf.keras.models.load_model(os.path.join(model_path, "simple_model.h5"))
    test_x = np.linspace(0, 2 * np.pi, 10)
    predictions = model.predict(test_x)

    mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
    mlflow.start_run()
    mlflow.log_metric("mean_prediction", np.mean(predictions))
    mlflow.log_artifact(model_path)
    mlflow.end_run()
    print("Predictions:", predictions)


# Create components from functions using create_component_from_func
create_dataset_component = create_component_from_func(create_dataset, packages_to_install=["numpy"])
preprocess_data_component = create_component_from_func(preprocess_data, packages_to_install=["numpy"])
train_model_component = create_component_from_func(train_model, packages_to_install=["tensorflow", "mlflow", "numpy"])
run_inference_component = create_component_from_func(run_inference, packages_to_install=["tensorflow", "numpy"])


# Define the pipeline using KFP v1 syntax with components created from functions
@dsl.pipeline(name="simple-ml-pipeline", description="Simple ML pipeline with PVCs and TensorBoard")
def simple_ml_pipeline():
    """
    Put your pipeline structure here.
    Define the volume and mount for shared data storage.
    Use the components created above and link them in the desired order.

    Example steps:
    - Create dataset (create_dataset_task)
    - Preprocess data (preprocess_data_task)
    - Train model (train_model_task)
    - Run inference (run_inference_task)
    """
    volume = k8s.V1Volume(
        name="shared-volume", persistent_volume_claim=k8s.V1PersistentVolumeClaimVolumeSource(claim_name="shared-pvc")
    )
    volume_mount = k8s.V1VolumeMount(name="shared-volume", mount_path="/mnt/shared")
    # Set MLflow tracking URI and MinIO environment variables
    MLFLOW_TRACKING_URI = "http://your-mlflow-server"
    MLFLOW_S3_ENDPOINT_URL = "http://your-minio-server"
    AWS_ACCESS_KEY_ID = "your-access-key"
    AWS_SECRET_ACCESS_KEY = "your-secret-key"
    os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_TRACKING_URI
    os.environ["MLFLOW_S3_ENDPOINT_URL"] = MLFLOW_S3_ENDPOINT_URL
    os.environ["AWS_ACCESS_KEY_ID"] = AWS_ACCESS_KEY_ID
    os.environ["AWS_SECRET_ACCESS_KEY"] = AWS_SECRET_ACCESS_KEY

    # Create dataset task
    create_dataset_task = create_dataset_component().add_volume(volume).add_volume_mount(volume_mount)

    # Preprocess dataset task
    preprocess_data_task = preprocess_data_component().add_volume(volume).add_volume_mount(volume_mount)
    preprocess_data_task.after(create_dataset_task)

    # Train model task
    train_model_task = train_model_component().add_volume(volume).add_volume_mount(volume_mount)
    train_model_task.after(preprocess_data_task)

    # Run inference task
    run_inference_task = run_inference_component().add_volume(volume).add_volume_mount(volume_mount)
    run_inference_task.after(train_model_task)


# Compile the pipeline into a .tar.gz or .yaml file (choose one)
kfp.compiler.Compiler().compile(simple_ml_pipeline, "simple_ml_pipeline.yaml")

1.8.22
